In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
import pickle

In [5]:
features = [
    'hour_of_day',
    'day_of_week_num',
    'is_break_period',
    'is_after_hours',
    'occupancy',
    'fan_status',
    'light_status',
    'power_consumption_w',
    'rolling_avg_30min',
    'lag_1h_power'
]

X = df[features]
y = df['wastage_label']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# 80% of data for training, 20% for testing

In [7]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 15]
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='f1')
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
print("Best parameters:", grid_search.best_params_)

Best parameters: {'max_depth': 5, 'n_estimators': 100}


In [8]:
y_pred = best_rf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 1.0


In [9]:
with open('../models/random_forest_model.pkl', 'wb') as f:
    pickle.dump(best_rf, f)
print("Model saved!")

Model saved!


In [10]:
from sklearn.ensemble import IsolationForest

# Use only the numeric feature columns (no labels needed)
X_anomaly = df[['power_consumption_w', 'rolling_avg_30min', 
                 'lag_1h_power', 'hour_of_day']]

iso_forest = IsolationForest(contamination=0.1, random_state=42)
iso_forest.fit(X_anomaly)

# -1 means anomaly, 1 means normal
df['anomaly_score'] = iso_forest.predict(X_anomaly)
print("Anomalies detected:", (df['anomaly_score'] == -1).sum())

Anomalies detected: 24


In [11]:
with open('../models/isolation_forest_model.pkl', 'wb') as f:
    pickle.dump(iso_forest, f)
print("Isolation Forest saved!")

Isolation Forest saved!


In [17]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Scale the data — LSTM works much better with values between 0 and 1
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(df[features + [target_col]].values)

sequence_length = 3
X_lstm, y_lstm = [], []
for i in range(sequence_length, len(data_scaled)):
    X_lstm.append(data_scaled[i-sequence_length:i, :-1])
    y_lstm.append(data_scaled[i, -1])

X_lstm = np.array(X_lstm)
y_lstm = np.array(y_lstm)
print("Total sequences:", len(X_lstm))

Total sequences: 247


In [18]:
n = len(X_lstm)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_tr, y_tr   = X_lstm[:train_end], y_lstm[:train_end]
X_val, y_val = X_lstm[train_end:val_end], y_lstm[train_end:val_end]
X_te, y_te   = X_lstm[val_end:], y_lstm[val_end:]

print(f"Train: {len(X_tr)}, Val: {len(X_val)}, Test: {len(X_te)}")

Train: 172, Val: 37, Test: 38


In [19]:
model = Sequential([
    LSTM(64, input_shape=(sequence_length, X_lstm.shape[2]), return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.summary()

# EarlyStopping stops training automatically when it stops improving
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=100,         # more epochs
    batch_size=8,       # smaller batch size for small datasets
    callbacks=[early_stop],
    verbose=1
)

B:\Tools\Anaconda\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Layer (type)               ┃ Output Shape        ┃    Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ lstm_1 (LSTM)              │ (None, 3, 64)       │     19,200 │
├────────────────────────────┼─────────────────────┼────────────┤
│ dropout (Dropout)          │ (None, 3, 64)       │          0 │
├────────────────────────────┼─────────────────────┼────────────┤
│ lstm_2 (LSTM)              │ (None, 32)          │     12,416 │
├────────────────────────────┼─────────────────────┼────────────┤
│ dropout_1 (Dropout)        │ (None, 32)          │          0 │
├────────────────────────────┼─────────────────────┼────────────┤
│ dense_2 (Dense)            │ (None, 16)          │        528 │
├────────────────────────────┼─────────────────────┼────────────┤
│ dense_3 (Dense)            │ (None, 1)           │         17 │
└────────────────────────────┴─────────────────────┴────────────┘

 Total params: 32,161 (125.63 KB)

 Trainable params: 32,161 (125.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 33ms/step - loss: 0.2487 - val_loss: 0.1260
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.1172 - val_loss: 0.1243
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1130 - val_loss: 0.1232
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.1119 - val_loss: 0.1217
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.1096 - val_loss: 0.1214
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.1145 - val_loss: 0.1292
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.1111 - val_loss: 0.1234
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.1058 - val_loss: 0.1214
Epoch 9/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.1113 - val_loss: 0.1243
Epoch 10/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0981 - val_loss: 0.1208
Epoch 11/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.1073 - val_loss: 0.1221
Epoch 12/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step -

In [20]:
import pickle

model.save('../models/lstm_model.keras')

# Save the scaler — you need it to reverse the scaling later
with open('../models/lstm_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Model and scaler saved!")

Model and scaler saved!
